# Market Pricing and Simplified IPV

Price a SPY option with real market data (yfinance) and perform a basic
Independent Price Verification against the bid/ask mid.

Data: SPY spot, call chain, implied volatility.
No synthetic prices. Reserve concepts (FairVA, P&L attribution) are out of scope.

In [2]:
import sys, json
sys.path.append("..")
import numpy as np
import pandas as pd
from pricing_engine.market_data import fetch_spy
from pricing_engine.black_scholes import black_scholes

snap = fetch_spy()
print(f"Spot SPY        : {snap['spot']:.2f}")
print(f"Expiration      : {snap['expiration']}")
print(f"Contrats propres: {len(snap['chain'])}")
print(f"Vol réalisée 30j: {snap['realized_vol']:.2%}")

Spot SPY        : 716.70
Expiration      : 2026-11-27
Contrats propres: 139
Vol réalisée 30j: 26.56%


## 1. Pricing with market implied volatility

Repricing with the market IV should reproduce the mid price: this validates
the Black-Scholes implementation (`sanity_gap` ≈ 0).

In [3]:
row = snap["chain"].iloc[len(snap["chain"]) // 2]   # contract near the money
S, K = snap["spot"], float(row["strike"])
T, r, q = float(row["T"]), 0.045, 0.0
iv = float(row["impliedVolatility"])

price_iv = black_scholes(S, K, r, iv, T, q, True)
mid = float(row["mid"])

print(f"Strike {K:.0f} | Mid {mid:.4f} | Prix avec IV marché {price_iv:.4f}")
print(f"Sanity gap : {price_iv - mid:+.4f}")

Strike 697 | Mid 20.6050 | Prix avec IV marché 20.0516
Sanity gap : -0.5534


## 2. Simplified IPV against market mid

Our independent mark uses realized volatility (not the market IV).
We flag a breach when the absolute difference exceeds max(0.05, 1% of mid).

In [4]:
rows = []
for _, c in snap["chain"].iterrows():
    K_i = float(c["strike"])
    mid_i = float(c["mid"])
    mark = black_scholes(S, K_i, r, snap["realized_vol"], float(c["T"]), q, True)
    diff = mark - mid_i
    tol = max(0.05, 0.01 * mid_i)
    rows.append({"strike": K_i, "mid": mid_i, "mark": mark,
                 "diff": diff, "tolerance": tol, "breach": abs(diff) > tol})

ipv = pd.DataFrame(rows)
print(ipv.head(10).to_string(index=False))
print(f"\nBreaches : {int(ipv['breach'].sum())} / {len(ipv)}")

 strike     mid       mark      diff  tolerance  breach
  500.0 216.940 216.761652 -0.178348    2.16940   False
  505.0 211.940 211.762269 -0.177731    2.11940   False
  510.0 206.940 206.762885 -0.177115    2.06940   False
  515.0 201.950 201.763501 -0.186499    2.01950   False
  520.0 196.945 196.764118 -0.180882    1.96945   False
  540.0 176.900 176.766583 -0.133417    1.76900   False
  545.0 171.950 171.767200 -0.182800    1.71950   False
  550.0 166.950 166.767816 -0.182184    1.66950   False
  555.0 161.955 161.768433 -0.186567    1.61955   False
  575.0 141.945 141.770898 -0.174102    1.41945   False

Breaches : 63 / 139


## 3. Save

In [6]:
with open("../models/market_pricing.json", "w") as f:
    json.dump({
        "source": snap["source"],
        "as_of": str(pd.Timestamp.today().date()),
        "spot": round(S, 2),
        "strikes": len(ipv),
        "sanity_gap": round(float(price_iv - mid), 4),
        "ipv_breaches": int(ipv["breach"].sum()),
        "note": "Simplified IPV: realized-vol mark vs market mid. "
                "FairVA and full P&L attribution out of scope.",
    }, f, indent=2)
print("Saved models/market_pricing.json")

Saved models/market_pricing.json
